# 1. Load data

## 1a) Load the protein data

In [ ]:
library(tidyverse)

In [ ]:
# load the protein saveRDS(protein_matrix, here::here("data", "analysis_data", "ukb", "protein_data" ,"protein_data.rds"))
protein_matrix <- readRDS(here::here("data", "analysis_data", "ukb", "protein_data" ,"protein_data.rds"))

In [ ]:
# Check the number of unique subjects in the protein matrix
cat("Number of unique subjects in the protein matrix:", length(unique(protein_matrix$eid)), "\n")

In [ ]:
# check the number of unique subjects in the each visit
cat("Number of unique subjects in the each visit:")
table(protein_matrix$INS_INDEX)

In [ ]:
# only keep the baseline visit data
protein_data_baseline <- protein_matrix %>% filter(INS_INDEX == 0)

# drop the INS_INDEX column
protein_data_baseline <- protein_data_baseline %>% select(-INS_INDEX)

# save the protein data
saveRDS(protein_data_baseline , here::here("data", "analysis_data", "ukb", "protein_data", "protein_data_baseline.rds"))


## 1b) Load the genotype data

In [ ]:
# load the genotype data here::here("data", "analysis_data", "ukb", "visit_info", "UKB_geno.csv")
geno_data <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "UKB_geno.csv"))


In [ ]:
# show the number of unique subjects in the genotype data
cat("Number of unique subjects in the genotype data:", length(unique(geno_data$IID)), "\n")

In [ ]:
# check the overlap between the protein and genotype data
# show number of each genotype group
cat("Number of each genotype group:")
table(geno_data$Genotype_Group)


## 1c) Load the phenotype data


In [ ]:
pheno_data <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "cleaned_pheno.csv"), row.names = 1)

## 1d) Load the withdrawl data

In [ ]:
#w847687_20250818.csv
withdrawl_data <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "w847687_20250818.csv"), header = FALSE)
length(withdrawl_data$V1)

# 2. Baseline visit selection
Since all ALS patients are only included in the baseline visit, we will only keep the subjects that have baseline visit pheno data and protein data.

## 2a) Select the subjects that with baseline visit
In this step, we will check the number of subjects that have baseline visit pheno data and protein data.


In [ ]:
# optional cleanup
baseline_id <- protein_matrix %>% filter(INS_INDEX == 0) %>% pull(eid)
protein_id <- unique(baseline_id)
pheno_id   <- unique(na.omit(pheno_data$eid))

cat("Raw number of protein data:", length(protein_id), "\n")
cat("Raw number of pheno data:", length(pheno_id), "\n")

common_id <- intersect(protein_id, pheno_id)
cat("Number of protein data with baseline visit pheno data:", length(common_id), "\n")

# 2b) Select the subjects that have available genotype data
In this step, we will check the number of subjects that have available genotype data. And will keep the subjects that have available genotype data.

In [ ]:
geno_id <- unique(na.omit(geno_data$IID))
cat("Raw number of geno data:", length(geno_id), "\n")

# show how many subjects do not have genotype data
cat("Number of subjects that do not have genotype data:", length(setdiff(common_id, geno_id)), "\n")

common_id <- intersect(common_id, geno_id)
cat("Number of protein data with baseline visit pheno data and genotype data:", length(common_id), "\n")


## 2c) Select the subjects that have available withdrawl data
In this step, we will check the number of subjects that have available withdrawl data. And will keep the subjects that have available withdrawl data.


In [ ]:
withdrawl_data <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "w847687_20250818.csv"), header = FALSE)
withdrawl_id <- unique(na.omit(withdrawl_data$V1))
cat("Raw number of withdrawl data:", length(withdrawl_id), "\n")

In [ ]:
lost_followup_id <- pheno_data %>% filter(!is.na(lost_date	)) %>% pull(eid)
cat("Number of subjects that have available withdrawl data:", length(lost_followup_id), "\n")

In [ ]:
total_withdrawl_id <- unique(c(lost_followup_id, withdrawl_id))

# show how many subjects withdrawl from the study
cat("Number of subjects that withdrawl from the study:", length(total_withdrawl_id), "\n")

# show how many subjects withdrawl from the study are in the protein data
cat("Number of subjects that withdrawl from the study are in the protein data:", length(intersect(common_id, total_withdrawl_id)), "\n")

common_id <- setdiff(common_id, total_withdrawl_id)
cat("Number of protein data with baseline visit pheno data and genotype data and withdrawl data:", length(common_id), "\n")

# 3. ALS result based selection
In this step, we will check the number of subjects that have available ALS result data. And will keep the subjects that have available ALS result data.

## 3a) Divide the subjects based on the diagnosis result with G12.2 or not

ALS and non-ALS based on the diagnosis result with G12.2 or not. 

In [ ]:
pheno_cleaned <- pheno_data %>% 
                filter(eid %in% common_id) %>% 
                left_join(geno_data, by = c("eid" = "IID"))
cat("Number of subjects that have available ALS result data:", dim(pheno_cleaned)[1], "\n")
cat("People with ALS:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS == 1])), "\n")
cat("People without ALS:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS == 0])), "\n")

## 3b) ALS group

Only keep the confirmed ALS cases, and divide the confirmed ALS cases into the Phenoconverter and 	Clinically Manifest ALS by their YrSinceDi
 


In [ ]:
# show the source of the G12 in the ALS == 1 group  
cat("Source of the G12:")
table(pheno_cleaned$G12_source[pheno_cleaned$ALS == 1])


In [ ]:
# Mutate a ALS_confirmed label for the people who have G12_source in "Primary care and other source(s)" ,"Hospital admissions data only", "Hospital admissions data and other source(s)"
pheno_cleaned <- pheno_cleaned %>% 
mutate(ALS_confirmed = ifelse(ALS == 1 & G12_count ==1 & G12_source %in% c("Primary care and other source(s)", 
                                                "Hospital admissions data only", 
                                                "Hospital admissions data and other source(s)"), 1, 0))
                                            
# show the number of ALS_confirmed == 1
cat("Number of ALS_confirmed == 1:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS_confirmed == 1])), "\n")


In [ ]:
# show the ALS_confirmed == 1 group and if the YrSinceDi is larger than 0
cat("Number of ALS_confirmed == 1 and YrSinceDi is larger than 0:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS_confirmed == 1 & pheno_cleaned$YrSinceDi > 0])), "\n")

# show the ALS_confirmed == 1 group and if the YrSinceDi is 0
cat("Number of ALS_confirmed == 1 and YrSinceDi is less than 0:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS_confirmed == 1 & pheno_cleaned$YrSinceDi < 0])), "\n")

## 3c) none ALS group

Divide the none ALS group into the control group and pre-ALS group.
Process:
1. Only keep the people who do not have any diseases of the nervous system (ICD-10=G00-G99) in the diagnosis result
2. Divide the group based on the genotype data, only focus on the SOD1 and C9orf72 genotypes
   - For people without any pathogenic variants, they are the control group
   - For people with pathogenic variants, they are the pre-ALS group


In [ ]:
#  load the data/analysis_data/ukb/visit_info/pheno_G00_G99_v1.csv
pheno_G00_G99 <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "pheno_G00_G99_v1.csv"))
pheno_G00_G99$has_G00_G99_record <- apply(
  pheno_G00_G99[, -c(1,2)],   # remove index and eid columns
  1,
  function(x) any(!is.na(x) & x != "")
)

pheno_G00_G99_id <- pheno_G00_G99 %>%
  filter(has_G00_G99_record == TRUE) %>%
  pull(eid)
  

In [ ]:
# Mutate a nervous system free label for the people without ALS
pheno_cleaned <- pheno_cleaned %>% mutate(G00_G99_id = ifelse(eid %in% pheno_G00_G99_id, 1, 0), 
    nervous_system_free = ifelse(ALS == 0 & G00_G99 == 0 & G00_G99_id == 0, 1, 0))

# say how many people are nervous system free in the none ALS group
cat("Number of people who are nervous system free in the none ALS group:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS == 0 & pheno_cleaned$nervous_system_free == 1])), "\n")
cat("Number of people who are with nervous system free in the none ALS group:", length(unique(pheno_cleaned$eid[pheno_cleaned$ALS == 0 & pheno_cleaned$nervous_system_free == 0])), "\n")

In [ ]:
# Among the none ALS and nervous system free group, show genotype distribution
cat("Genotype distribution in the none ALS and nervous system free group:")
table(pheno_cleaned$Genotype_Group[pheno_cleaned$ALS == 0 & pheno_cleaned$nervous_system_free == 1])

# 4. Build the group label



In [ ]:
pheno_final <- pheno_cleaned %>%
  mutate(
    group = case_when(
      # ALS == 1 logic
      ALS == 1 & ALS_confirmed == 1 & YrSinceDi > 0 ~ "Clinically manifest ALS",

      ALS == 1 & ALS_confirmed == 1 & YrSinceDi > -2 & YrSinceDi < 0 ~ "Pre-hospital",
      ALS == 1 & ALS_confirmed == 1 & YrSinceDi <= -2 ~ "Phenoconverter",
      ALS == 1 & ALS_confirmed == 0 ~ "drop",

      # ALS == 0 logic
      ALS == 0 & nervous_system_free == 0 ~ "drop",
      ALS == 0 & nervous_system_free == 1 & Genotype_Group %in% c("SOD1 nonA4V", "C9orf72") ~ "Pre-symptomatic",
      ALS == 0 & nervous_system_free == 1 & Genotype_Group == "None identified" ~ "Healthy control",
      ALS == 0 & nervous_system_free == 1 & Genotype_Group == "Other Genotype" ~ "drop",

      TRUE ~ NA_character_
    )
  )

# Update the other genotype genotype in Convert , only keep the other_status_als is 1 to be the other genotype, otherwise genotype move to none identifie
pheno_final <- pheno_final %>%
  mutate(
    Genotype_Group = case_when(
      group == "Phenoconverter" & Genotype_Group == "Other Genotype" & other_status_als == 1 ~ "Other Genotype",
      group == "Phenoconverter" & Genotype_Group == "Other Genotype" & other_status_als != 1 ~ "None identified",
      TRUE ~ Genotype_Group
    )
  )

# set the order of the group
pheno_final <- pheno_final %>% mutate(group = factor(group, levels = c("Healthy control", "Pre-symptomatic", "Phenoconverter","Pre-hospital", "Clinically manifest ALS", "drop")))

# set the order of the Genotype_Group
pheno_final <- pheno_final %>% mutate(Genotype_Group = factor(Genotype_Group, levels = c("SOD1 nonA4V", "C9orf72","Other Genotype","None identified")))

cat("Number of each group:\n")
print(table(pheno_final$group, pheno_final$Genotype_Group, useNA = "ifany"))


# keep the select columns
# eid, sex, age, Genotype_Group, group, ALS, YrSinceDi, YrSinceCen
pheno_final <- pheno_final %>% select(eid, sex, age, Genotype_Group, group, ALS, YrSinceDi, YrSinceCen)

# use the same name as the miamiNow  group instead of Group, age instead of CollAge, Genotype_Group instead of GenoGroup, etc.).
pheno_final <- pheno_final %>% rename(Group = group, CollAge = age, GenoGroup = Genotype_Group, Sex = sex)

# save the rows without group == "drop"
pheno_final <- pheno_final %>% filter(Group != "drop")

# show the shape of the pheno_cleaned data


In [ ]:
table(pheno_final$Group)

In [ ]:
cat("Shape of the pheno_cleaned data:", dim(pheno_final), "\n")

# save the pheno_cleaned data csv file
write.csv(pheno_final, here::here("data", "analysis_data", "ukb", "visit_info", "visit_info.csv"))
